In [1]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from io import StringIO  # Added import
import time
import random

# Configure Chrome options for Brave browser
chrome_options = Options()
chrome_options.binary_location = "C:/Program Files/BraveSoftware/Brave-Browser/Application/brave.exe"
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--window-size=1920,1080")
# chrome_options.add_argument("--headless=new")  # Uncomment for headless mode

def scrape_league_results(url, league_name):
    driver = None
    try:
        # Initialize driver with version matching Brave's Chromium
        driver = webdriver.Chrome(
            service=Service(ChromeDriverManager(driver_version="136.0.7103").install()),
            options=chrome_options
        )
        
        # Load page with randomized delays
        driver.get(url)
        time.sleep(random.uniform(2, 5))
        
        # Extract rendered HTML
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table = soup.find('table', {'id': 'sched_2023-2024_9_1'}) or soup.find('table', class_='stats_table')
        
        if not table:
            print(f"⚠️ Table not found for {league_name}")
            return None
        
        # Parse table data with StringIO wrapper
        df = pd.read_html(StringIO(str(table)))[0]  # Fixed line
        
        # Clean and transform data
        df = df.dropna(subset=['Score'])
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df[['Home_Score', 'Away_Score']] = df['Score'].str.split('–', expand=True)
        df = df.rename(columns={
            'Home': 'Home_Team',
            'Away': 'Away_Team',
            'xG': 'Home_xG',
            'xG.1': 'Away_xG'
        })[['Date', 'Home_Team', 'Away_Team', 'Home_Score', 'Away_Score', 'Home_xG', 'Away_xG']]
        df['League'] = league_name
        return df
    
    except Exception as e:
        print(f"❌ Failed to scrape {league_name}: {str(e)}")
        return None
    finally:
        if driver:
            driver.quit()

def scrape_all_leagues():
    leagues = {
        'Premier League': 'https://fbref.com/en/comps/9/schedule/Premier-League-Scores-and-Fixtures',
        'La Liga': 'https://fbref.com/en/comps/12/schedule/La-Liga-Scores-and-Fixtures',
        'Bundesliga': 'https://fbref.com/en/comps/20/schedule/Bundesliga-Scores-and-Fixtures',
        'Ligue 1': 'https://fbref.com/en/comps/13/schedule/Ligue-1-Scores-and-Fixtures',
        'Serie A': 'https://fbref.com/en/comps/11/schedule/Serie-A-Scores-and-Fixtures',
        'Serie B': 'https://fbref.com/en/comps/18/schedule/Serie-B-Scores-and-Fixtures',
        'Championship': 'https://fbref.com/en/comps/10/schedule/Championship-Scores-and-Fixtures'
    }

    dataframes = []
    for league_name, url in leagues.items():
        print(f"\n🔍 Scraping {league_name}...")
        df = scrape_league_results(url, league_name)
        if df is not None:
            print(f"✅ Success: {len(df)} matches found")
            dataframes.append(df)
        time.sleep(random.uniform(5, 15))

    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        combined_df.to_csv('Fixture_Results.csv', index=False)
        print(f"\n🎉 Successfully saved {len(combined_df)} matches!")
        return combined_df
    else:
        print("\n💥 All scraping attempts failed. Check logs for details.")
        return None

if __name__ == "__main__":
    scrape_all_leagues()


🔍 Scraping Premier League...
✅ Success: 390 matches found

🔍 Scraping La Liga...
✅ Success: 391 matches found

🔍 Scraping Bundesliga...
✅ Success: 323 matches found

🔍 Scraping Ligue 1...
✅ Success: 323 matches found

🔍 Scraping Serie A...
✅ Success: 390 matches found

🔍 Scraping Serie B...
✅ Success: 405 matches found

🔍 Scraping Championship...
✅ Success: 586 matches found

🎉 Successfully saved 2808 matches!
